# Logistic Regression, Explained Simply
Predicting yes or no with probabilities: the sigmoid, log loss and decision boundaries. Runs free in Google Colab.

Run each cell from top to bottom with **Shift + Enter**.

## 1. The data
200 students: hours studied, hours slept the night before, and whether they passed (1) or failed (0).

### 200 students
- `exam_results.csv`: 200 students: hours studied, hours slept, and passed (1 = yes, 0 = no)
- `value_counts().to_dict()`: how many passed (1) and how many failed (0)

In [ ]:
import pandas as pd

url = ("https://raw.githubusercontent.com/Uttam-Nepal/"
       "machine-learning-explained/main/"
       "logistic-regression/exam_results.csv")
df = pd.read_csv(url)
print(df.shape)
print(df["passed"].value_counts().to_dict())
df.head()

### Hours studied against pass or fail
- `the picture`: two rows of dots: fails at 0, passes at 1; more study, more passes

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(df["hours_studied"], df["passed"], alpha=0.4)
plt.xlabel("hours studied")
plt.ylabel("passed (1) or failed (0)")
plt.show()

## 2. The sigmoid
**sigmoid(z) = 1 / (1 + e^-z)** squashes any number into a probability between 0 and 1. e is a special number, about 2.718.

### The sigmoid function
- `np.exp(-z)`: e to the power of minus z
- `sigmoid(0)`: exactly 0.5: a coin flip
- `big positive z`: close to 1
- `big negative z`: close to 0

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

for z in [-6, -2, 0, 2, 6]:
    print(f"sigmoid({z:>2}) = {sigmoid(z):.3f}")

### The S-shaped curve
- `np.linspace(-8, 8, 200)`: 200 values of z from -8 to 8
- `the shape`: an S: flat near 0, rising through 0.5, flat near 1

In [ ]:
z = np.linspace(-8, 8, 200)
plt.plot(z, sigmoid(z))
plt.axhline(0.5, color="gray", linestyle="--")
plt.xlabel("z"); plt.ylabel("sigmoid(z)")
plt.title("The sigmoid squashes any number into 0 to 1")
plt.show()

## 3. Logistic regression with one input
**z = w x hours + b**, then **probability = sigmoid(z)**. If the probability is at least 0.5, predict pass.

### Fit the model
- `LogisticRegression()`: the model
- `coef_ and intercept_`: w and b
- `-b / w`: the hours where z = 0, so the probability is exactly 0.5: the decision boundary

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

train, test = train_test_split(df, test_size=0.25, random_state=2)
model = LogisticRegression()
model.fit(train[["hours_studied"]], train["passed"])
w, b = model.coef_[0][0], model.intercept_[0]
print(f"w = {w:.2f}, b = {b:.2f}")
print(f"boundary: {-b / w:.1f} hours")

### Probabilities for different study times
- `predict_proba`: a probability for each class: column 0 is fail, column 1 is pass
- `[:, 1]`: keep just the pass probabilities
- `:.0%`: show as a percentage

In [ ]:
hours = pd.DataFrame({"hours_studied": [1, 3, 5, 7, 9]})
probs = model.predict_proba(hours)[:, 1]
for h, p in zip(hours["hours_studied"], probs):
    print(f"{h} hours: {p:.0%} chance of passing")

### The fitted S-curve
- `the red curve`: the pass probability for every study time
- `the dashed line`: the decision boundary, where the probability is 0.5

In [ ]:
grid = pd.DataFrame({"hours_studied": np.linspace(0, 10, 200)})
plt.scatter(df["hours_studied"], df["passed"], alpha=0.3)
plt.plot(grid, model.predict_proba(grid)[:, 1], color="red")
plt.axvline(-b / w, color="gray", linestyle="--")
plt.xlabel("hours studied"); plt.ylabel("probability of passing")
plt.show()

### How often is it right?
- `model.predict`: pass (1) if the probability is at least 0.5, otherwise fail (0)
- `accuracy_score`: the share of test students it got right

In [ ]:
from sklearn.metrics import accuracy_score

pred = model.predict(test[["hours_studied"]])
print("accuracy:", round(accuracy_score(test["passed"], pred), 2))

## 4. Log loss
Logistic regression is trained to minimise **log loss**: for each student, minus the log of the probability it gave to what really happened.

### Log loss for a few predictions
- `-np.log(p)`: minus the natural log of the probability given to the true answer
- `the output`: confident and right costs almost nothing; confident and wrong costs a lot

In [ ]:
for p in [0.99, 0.9, 0.5, 0.1, 0.01]:
    print(f"gave the true answer {p:.0%}  ->  loss {-np.log(p):.2f}")

## 5. Two inputs: a decision boundary line
With hours studied and hours slept, **z = w1 x studied + w2 x slept + b**, and the boundary z = 0 is a straight line.

### Two inputs
- `two weights`: one for study, one for sleep: both positive, both help
- `accuracy`: better than with study alone

In [ ]:
features = ["hours_studied", "hours_slept"]
model2 = LogisticRegression().fit(train[features], train["passed"])
print("weights:", model2.coef_[0].round(2))
print("intercept:", model2.intercept_[0].round(2))
pred2 = model2.predict(test[features])
print("accuracy:", round(accuracy_score(test["passed"], pred2), 2))

### Draw the boundary
- `map({0: "red", 1: "green"})`: colour each student by the result
- `-(w1 * xs + b2) / w2`: where z = 0: solve w1 x studied + w2 x slept + b = 0 for slept
- `the line`: above and right: predicted pass; below and left: predicted fail

In [ ]:
w1, w2 = model2.coef_[0]; b2 = model2.intercept_[0]
colors = df["passed"].map({0: "red", 1: "green"})
plt.scatter(df["hours_studied"], df["hours_slept"], c=colors, alpha=0.6)
xs = np.array([0, 10])
plt.plot(xs, -(w1 * xs + b2) / w2, color="black")
plt.ylim(4, 9); plt.xlabel("hours studied"); plt.ylabel("hours slept")
plt.title("green = passed, red = failed"); plt.show()

## 6. Mini project: a pass predictor
Give a study plan, get a probability and advice.

### Pass predictor
- `predict_proba(student)[0, 1]`: the pass probability for this one student
- `advise(*plan)`: the star unpacks (2, 5) into studied=2, slept=5

In [ ]:
def advise(studied, slept):
    student = pd.DataFrame([[studied, slept]], columns=features)
    p = model2.predict_proba(student)[0, 1]
    verdict = "likely pass" if p >= 0.5 else "at risk"
    return f"{studied} h study, {slept} h sleep: {p:.0%} ({verdict})"

for plan in [(2, 5), (4, 8), (7, 6)]:
    print(advise(*plan))

## Try it yourself
1. What pass probability does the two-input model give 3 hours of study and 9 hours of sleep?
2. Change the threshold to 0.7. How many test students are predicted to pass now?
3. Work out sigmoid(2) by hand: 1 / (1 + e^-2). Check it with the sigmoid function.